# Find best hyperparams for each mouse via Hoffman job array

In [ ]:
from glm_hmm_utils.batch_utils import create_params_array, build_submission_script, stitch_outputs
import subprocess
import numpy as np
import os
%matplotlib inline
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['pdf.fonttype'] = 42
matplotlib.rcParams['ps.fonttype'] = 42

from pathlib import Path

FIGPATH = Path('/u/home/m/mmelin/state_figs')
SAVING_FIGS = False

SAVEPATH = '/u/home/m/mmelin/UGE_GLM_runs/fez_mice_crossval'

#ANIMALS = [['mSM63','mSM64','mSM65','mSM66']]
ANIMALS = [['Fez71','Fez72','Fez73','Fez74','Fez75']]

PYFILEPATH = os.path.join(os.getcwd(), 'cluster_crossval.py')

METHODS = ['map','mle']
N_ITERS = np.array([2_000])
N_STATES = [1, 2, 3, 4, 5, 6]
ALPHAS = np.array([1,2])
SIGMAS = np.array([.25, 0.5, 0.75, 1])

### Create the paramater matrix which is read out by the array jobs

In [ ]:
numjobs, numparams, _ = create_params_array(SAVEPATH, ANIMALS, METHODS, N_ITERS, N_STATES, ALPHAS, SIGMAS)
print(f'There are {numjobs} jobs to be run with {numparams} params.')

### Submit the job

In [ ]:
fname = build_submission_script(SAVEPATH, 'submit.sh', PYFILEPATH, numjobs, pyfile_args=[SAVEPATH], n_cpu_per_job=20)
#fname = build_submission_script(SAVEPATH, 'submit.sh', PYFILEPATH, numjobs, pyfile_args=[SAVEPATH], n_cpu_per_job=6)

In [ ]:
cmd = ['qsub', fname]
print(cmd)
subprocess.check_output(cmd)

## Rebuild the outputs and plot results

In [ ]:
param_matrix = np.load(os.path.join(SAVEPATH,'params.npy'), allow_pickle=True)
output = stitch_outputs(SAVEPATH)
output = np.array(output)
print(len(output))
param_matrix.shape

In [ ]:
valid = [o is not None for o in output]
param_matrix = param_matrix[valid,:]
output = output[valid]
output = np.stack(output)

In [ ]:
n_states = np.unique(param_matrix[:,3])
animals = np.unique(param_matrix[:,0])

full = []
for mouse in animals:
    onemouse = []
    for statenum in n_states:
        arr = np.empty((len(animals),len(n_states)))
        #mouse_inds = param_matrix[:,0] == mouse
        mouse_inds = np.array([par[0] == mouse for par in param_matrix])
        state_inds = np.array([par[3] == statenum for par in param_matrix])
        #state_inds = param_matrix[:,3] == statenum
        temp = output[mouse_inds & state_inds]  
        small_params_matrix = param_matrix[mouse_inds & state_inds,:]
        avg_cv_ll = [np.nanmean(t) for t in temp if t is not None]
        #avg_cv_ll = np.nanmean(temp, axis=1) # get average cv_ll over folds for one mouse
        if avg_cv_ll == []:
            max_cv_ll = None
        else:
            max_cv_ll = np.nanmax(avg_cv_ll)
            best_model_ind = np.argmax(avg_cv_ll)
            #max_cv_ll = np.nanmean(avg_cv_ll)
            print(f'Parameters for best model fit: {small_params_matrix[best_model_ind,:]}')
        onemouse.append(max_cv_ll)
    full.append(onemouse)
    
ll = np.stack(full)

In [ ]:
def separate_axes(ax):
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    yti = ax.get_yticks()
    yti = yti[(yti >= ax.get_ylim()[0]) & (yti <= ax.get_ylim()[1]+10**-4)] #Add a small value to cover for some very tiny added values
    ax.spines['left'].set_bounds([yti[0], yti[-1]])
    xti = ax.get_xticks()
    xti = xti[(xti >= ax.get_xlim()[0]) & (xti <= ax.get_xlim()[1]+10**-4)]
    ax.spines['bottom'].set_bounds([xti[0], xti[-1]])
    return

In [ ]:
#plotting
fig, axs = plt.subplots(4,4, figsize=(12,9))

for ix,ax in zip(range(len(ll)),np.ravel(axs)):
    ax.plot(n_states, ll[ix], color='black')
    ax.scatter(n_states, ll[ix], s=8, color='black')
    ax.set_title(animals[ix])
    ax.set_xlabel('# states')
    ax.set_ylabel('Cross-validated log likelihood')
    ax.set_xticks(n_states.astype(int))
    #separate_axes(ax)

fig.tight_layout()
fig.show()
if SAVING_FIGS:
    plt.savefig(FIGPATH / 'crossval_states.pdf', dpi=500, format='pdf', bbox_inches='tight')


In [ ]:
#plot distributions of N states here
#mean_ll = np.mean(output, axis=1)
#mean_ll = [np.mean(a) if a is not None else None for a in output]
mean_ll = np.mean(output, axis=1)
for mouse in ANIMALS:
    for state in N_STATES:
        is_mouse = [p == mouse for p in param_matrix[:,0]]
        inds = np.logical_and(is_mouse, param_matrix[:,3].astype('int') == state)
        print(inds.sum())
        results = mean_ll[inds]
        
        plt.hist(results, bins=10,label=f'{state} states')
    plt.legend()
    plt.xlabel('Log likelihood')
    plt.ylabel('Number of jobs')
    plt.title(f'{mouse}: cross-validation over different n_states')
    plt.show()


In [ ]:
#plot distributions over N_ITERS
for mouse in ANIMALS:
    for iters in N_ITERS:
        results = mean_ll[np.logical_and(param_matrix[:,0] == mouse, param_matrix[:,2].astype('int') == iters)]
        
        plt.hist(results, bins=100,label=f'{iters} iterations')
    plt.legend()
    plt.xlabel('Log likelihood')
    plt.ylabel('Number of jobs')
    plt.title(f'{mouse}: cross-validation over different iteration number')
    plt.show()

In [ ]:
STATE = 4
for mouse in ANIMALS:
    for alpha in ALPHAS:
        mask = np.logical_and.reduce([param_matrix[:,0] == mouse, param_matrix[:,4].astype('int') == alpha, param_matrix[:,3].astype('int') == STATE])
        results = mean_ll[mask]
        plt.hist(results, bins=50,label=f'Alpha = {alpha}')
    plt.legend()
    plt.xlabel('Log likelihood')
    plt.ylabel('Number of jobs')
    plt.title(f'{mouse}: cross-validation over different alphas')
    plt.show()

In [ ]:
STATE = 4
for mouse in ANIMALS:
    for sigma in SIGMAS:
        mask = np.logical_and.reduce([param_matrix[:,0] == mouse, param_matrix[:,5].astype('float') == sigma, param_matrix[:,3].astype('int') == STATE])
        results = mean_ll[mask]
        plt.hist(results, bins=40,label=f'sigma = {sigma}')
    plt.legend()
    plt.xlabel('Log likelihood')
    plt.ylabel('Number of jobs')
    plt.title(f'{mouse}: cross-validation over different sigmas')
    plt.show()